In [1]:
!pip install -q langgraph langchain langchain-openai langchain-chroma chromadb langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61

In [2]:
import os
os.environ["OPENAI_API_KEY"] = "ضع_مفتاح_OpenAI_هنا"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "ضع_مفتاح_LangSmith_هنا"
os.environ["LANGCHAIN_PROJECT"] = "capstone-agentic-assistant"

In [35]:
import os
os.environ["GROQ_API_KEY"] = "ضع_مفتاحك_هنا"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "ضع_مفتاح_LangSmith_هنا"
os.environ["LANGCHAIN_PROJECT"] = "capstone-agentic-assistant"

In [4]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("قول لي 'مرحبا' بس، رد قصير")
print(response.content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be3-8871-7e32-8332-6790dd9915bf,id=019f8be3-8871-7e32-8332-6790dd9915bf


مرحباً!


In [27]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)print(llm.invoke("قول مرحبا").content)

SyntaxError: invalid syntax (2172978662.py, line 2)

In [7]:
from langchain_core.tools import tool

@tool
def get_calendar_events(date: str) -> str:
    """يرجع مواعيد اليوم المحدد."""
    fake_events = {
        "2026-07-24": ["اجتماع فريق 10ص", "مراجعة تقرير 2م"]
    }
    events = fake_events.get(date, [])
    return f"مواعيد {date}: {events}" if events else f"لا مواعيد في {date}"

llm_with_tools = llm.bind_tools([get_calendar_events])
result = llm_with_tools.invoke("وش مواعيدي يوم 2026-07-24؟")
print(result.tool_calls)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be4-f675-7fe0-a821-452e55a0f5ca,id=019f8be4-f675-7fe0-a821-452e55a0f5ca


[{'name': 'get_calendar_events', 'args': {'date': '2026-07-24'}, 'id': '3kk8398bw', 'type': 'tool_call'}]


In [8]:
!pip install -q langgraph

In [9]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

calendar_tools = [get_calendar_events]
llm_calendar = llm.bind_tools(calendar_tools)

def calendar_agent(state: MessagesState):
    return {"messages": [llm_calendar.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("agent", calendar_agent)
builder.add_node("tools", ToolNode(calendar_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

calendar_graph = builder.compile()

result = calendar_graph.invoke({"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]})
print(result["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2c76-7113-95c9-24ce7f671597; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2c7a-76c1-9f16-5d62ee2ba8f8; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2c7b-7a53-89cf-c5bf5b46aac4
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2d67-7bf0-bf1e-f817562ad8d3; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2d69-7ca3-bec5-33b90b3e2321; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2d6c-7b22-90df-4a8b9652a01b; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2d6e-7a02-bbaf-2811248c87ca; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2d6f-7413-b7af-4ca54a02bc00; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2c7a-76c1-9f16-5d62ee2ba8f8; trace=019f8be7-2c76-7113-95c9-24ce7f671597,id=019f8be7-2c7b-7a53-89cf-c5bf5b46aac4
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=

مواعيدك في يوم 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


In [10]:
@tool
def send_email(to: str, subject: str) -> str:
    """يرسل إيميل لشخص معين بعنوان محدد."""
    return f"تم إرسال إيميل إلى {to} بعنوان: {subject}"

email_tools = [send_email]
llm_email = llm.bind_tools(email_tools)

def email_agent(state: MessagesState):
    return {"messages": [llm_email.invoke(state["messages"])]}

def supervisor(state: MessagesState) -> str:
    last_msg = state["messages"][-1].content.lower()
    if "ايميل" in last_msg or "إيميل" in last_msg or "email" in last_msg:
        return "email_agent"
    return "calendar_agent"

In [11]:
def route(state: MessagesState):
    return supervisor(state)

builder2 = StateGraph(MessagesState)
builder2.add_node("calendar_agent", calendar_agent)
builder2.add_node("email_agent", email_agent)
builder2.add_node("calendar_tools", ToolNode(calendar_tools))
builder2.add_node("email_tools", ToolNode(email_tools))

builder2.add_conditional_edges(START, route, {"calendar_agent": "calendar_agent", "email_agent": "email_agent"})

builder2.add_conditional_edges("calendar_agent", tools_condition, {"tools": "calendar_tools", END: END})
builder2.add_edge("calendar_tools", "calendar_agent")

builder2.add_conditional_edges("email_agent", tools_condition, {"tools": "email_tools", END: END})
builder2.add_edge("email_tools", "email_agent")

main_graph = builder2.compile()

r1 = main_graph.invoke({"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]})
print("Calendar:", r1["messages"][-1].content)

r2 = main_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]})
print("Email:", r2["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2212-72f3-8bc8-c40de5c45075; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2219-76a3-9c12-a373347a8f50; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221b-7880-8823-34f907f9e101; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221c-7482-86d0-16ae48906ead; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2225-77c3-badd-f6364e7df266
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a5-7b42-af1a-3a68ce8e36c0; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a9-7652-b5c0-159a71ec934b; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23af-7660-80d9-aa1254d77a81; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23bb-7fb3-aefe-d7b17e1ae34c; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23c4-7d41-8a60-cbe087e54b40; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-22

Calendar: مواعيدك في يوم 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-280e-7303-b327-759265d7d2ef; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2810-7111-966d-1f10e153c992; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2814-7871-9104-470230b341a7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281b-7802-879c-68d8afb8975d; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281e-7162-ae9b-734991aa9244; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b5-7782-87ca-d441e1d22333; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b2-71a1-98fc-8cd8bf7b3920
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2956-7f13-9193-dd3434536dcc; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295c-7500-957b-ed8345570631; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295d-7ae1-98c4-ab45b15c4dc7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-29

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ky5y0y8zf8hskxhxernv52dt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96925, Requested 3188. Please try again in 1m37.631999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
def route(state: MessagesState):
    return supervisor(state)

builder2 = StateGraph(MessagesState)
builder2.add_node("calendar_agent", calendar_agent)
builder2.add_node("email_agent", email_agent)
builder2.add_node("calendar_tools", ToolNode(calendar_tools))
builder2.add_node("email_tools", ToolNode(email_tools))

builder2.add_conditional_edges(START, route, {"calendar_agent": "calendar_agent", "email_agent": "email_agent"})

builder2.add_conditional_edges("calendar_agent", tools_condition, {"tools": "calendar_tools", END: END})
builder2.add_edge("calendar_tools", "calendar_agent")

builder2.add_conditional_edges("email_agent", tools_condition, {"tools": "email_tools", END: END})
builder2.add_edge("email_tools", "email_agent")

main_graph = builder2.compile()

r1 = main_graph.invoke({"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]})
print("Calendar:", r1["messages"][-1].content)

r2 = main_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]})
print("Email:", r2["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2212-72f3-8bc8-c40de5c45075; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2219-76a3-9c12-a373347a8f50; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221b-7880-8823-34f907f9e101; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221c-7482-86d0-16ae48906ead; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2225-77c3-badd-f6364e7df266
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a5-7b42-af1a-3a68ce8e36c0; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a9-7652-b5c0-159a71ec934b; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23af-7660-80d9-aa1254d77a81; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23bb-7fb3-aefe-d7b17e1ae34c; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23c4-7d41-8a60-cbe087e54b40; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-22

Calendar: مواعيدك في يوم 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-280e-7303-b327-759265d7d2ef; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2810-7111-966d-1f10e153c992; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2814-7871-9104-470230b341a7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281b-7802-879c-68d8afb8975d; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281e-7162-ae9b-734991aa9244; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b5-7782-87ca-d441e1d22333; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b2-71a1-98fc-8cd8bf7b3920
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2956-7f13-9193-dd3434536dcc; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295c-7500-957b-ed8345570631; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295d-7ae1-98c4-ab45b15c4dc7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-29

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ky5y0y8zf8hskxhxernv52dt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96925, Requested 3188. Please try again in 1m37.631999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
def route(state: MessagesState):
    return supervisor(state)

builder2 = StateGraph(MessagesState)
builder2.add_node("calendar_agent", calendar_agent)
builder2.add_node("email_agent", email_agent)
builder2.add_node("calendar_tools", ToolNode(calendar_tools))
builder2.add_node("email_tools", ToolNode(email_tools))

builder2.add_conditional_edges(START, route, {"calendar_agent": "calendar_agent", "email_agent": "email_agent"})

builder2.add_conditional_edges("calendar_agent", tools_condition, {"tools": "calendar_tools", END: END})
builder2.add_edge("calendar_tools", "calendar_agent")

builder2.add_conditional_edges("email_agent", tools_condition, {"tools": "email_tools", END: END})
builder2.add_edge("email_tools", "email_agent")

main_graph = builder2.compile()

r1 = main_graph.invoke({"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]})
print("Calendar:", r1["messages"][-1].content)

r2 = main_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]})
print("Email:", r2["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2212-72f3-8bc8-c40de5c45075; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2219-76a3-9c12-a373347a8f50; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221b-7880-8823-34f907f9e101; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-221c-7482-86d0-16ae48906ead; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-2225-77c3-badd-f6364e7df266
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a5-7b42-af1a-3a68ce8e36c0; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23a9-7652-b5c0-159a71ec934b; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23af-7660-80d9-aa1254d77a81; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23bb-7fb3-aefe-d7b17e1ae34c; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-23c4-7d41-8a60-cbe087e54b40; trace=019f8be9-2212-72f3-8bc8-c40de5c45075,id=019f8be9-22

Calendar: مواعيدك في يوم 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-280e-7303-b327-759265d7d2ef; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2810-7111-966d-1f10e153c992; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2814-7871-9104-470230b341a7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281b-7802-879c-68d8afb8975d; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-281e-7162-ae9b-734991aa9244; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b5-7782-87ca-d441e1d22333; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-26b2-71a1-98fc-8cd8bf7b3920
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-2956-7f13-9193-dd3434536dcc; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295c-7500-957b-ed8345570631; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-295d-7ae1-98c4-ab45b15c4dc7; trace=019f8be9-26ac-7c33-ae54-706549544e36,id=019f8be9-29

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ky5y0y8zf8hskxhxernv52dt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96925, Requested 3188. Please try again in 1m37.631999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [12]:
import time
from groq import RateLimitError

def invoke_with_retry(graph, input_data, max_retries=3):
    for attempt in range(max_retries):
        try:
            return graph.invoke(input_data)
        except RateLimitError as e:
            wait = 20 * (attempt + 1)
            print(f"Rate limit — retrying in {wait}s...")
            time.sleep(wait)
    raise Exception("فشل بعد عدة محاولات")

# جرب بعد شوي
r1 = invoke_with_retry(main_graph, {"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]})
print("Calendar:", r1["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-741c-7801-b555-a1fb08d27606; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-741e-7c03-80f5-006e0573267a; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-741f-7b43-873d-73b7261e5026; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7420-7331-97bd-72221c209a4a; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7421-79d2-8a9a-b38e0c706fdc
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7522-7991-8dc8-93347dd14b2f; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7526-7b43-9729-3e2f0b81c47d; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7529-7370-9b60-8f3397aac68a; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7533-74e0-af43-f61b0538b78f; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-7534-7090-8bab-42d978d4b383; trace=019f8bf4-741c-7801-b555-a1fb08d27606,id=019f8bf4-74

Calendar: مواعيدك في 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


In [13]:
!pip install -q sentence-transformers

In [16]:
!pip install -q langchain-text-splitters

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

policy_text = """
سياسة الإجازات:
يحق للموظف 30 يوم إجازة سنوية مدفوعة.
يجب تقديم طلب الإجازة قبل أسبوعين على الأقل من التاريخ المطلوب.
الإجازة المرضية تحتاج تقرير طبي بعد 3 أيام متتالية.

سياسة العمل عن بعد:
يمكن للموظف العمل من المنزل يومين بالأسبوع بعد موافقة المدير المباشر.
يجب الحضور للمكتب في الاجتماعات الأسبوعية.
"""

docs = [Document(page_content=policy_text)]
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)
print(f"عدد القطع: {len(chunks)}")

عدد القطع: 2


In [19]:
!pip install -q langchain-huggingface langchain-chroma

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# اختبار
results = retriever.invoke("كم يوم إجازة سنوية؟")
for r in results:
    print(r.page_content)
    print("---")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

سياسة الإجازات:
يحق للموظف 30 يوم إجازة سنوية مدفوعة.
يجب تقديم طلب الإجازة قبل أسبوعين على الأقل من التاريخ المطلوب.
الإجازة المرضية تحتاج تقرير طبي بعد 3 أيام متتالية.
---
سياسة العمل عن بعد:
يمكن للموظف العمل من المنزل يومين بالأسبوع بعد موافقة المدير المباشر.
يجب الحضور للمكتب في الاجتماعات الأسبوعية.
---


In [23]:
def invoke_with_retry(graph, input_data, config=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            if config:
                return graph.invoke(input_data, config=config)
            return graph.invoke(input_data)
        except RateLimitError as e:
            wait = 20 * (attempt + 1)
            print(f"Rate limit — retrying in {wait}s...")
            time.sleep(wait)
    raise Exception("فشل بعد عدة محاولات")

config = {"configurable": {"thread_id": "user-1"}}

r1 = invoke_with_retry(main_graph_with_memory, {"messages": [("user", "وش مواعيدي يوم 2026-07-24؟")]}, config=config)
print("رد 1:", r1["messages"][-1].content)

r2 = invoke_with_retry(main_graph_with_memory, {"messages": [("user", "شكراً، هل هذا يشمل يوم الأحد؟")]}, config=config)
print("رد 2:", r2["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0c2-7d50-97ae-919977e40cb1; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0c6-7a22-8697-4c62b7c271ee; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0c6-7a22-8697-4c7e18211dcf; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0c8-7b20-b4fe-179a63b31e93; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0c9-7882-b9c5-9232cda25be4
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a1ab-7290-aa32-ddeae2125775; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a1ad-73e0-994f-6efa27df0580; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a1ae-77b2-ac90-39ffd3d71acd; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a1b0-7f21-b4d3-00224ff2c935; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a1b0-7f21-b4d3-0037e341f051; trace=019f8bfc-a0c2-7d50-97ae-919977e40cb1,id=019f8bfc-a0

رد 1: مواعيدك في يوم 2024-07-24 هي:
- اجتماع فريق الساعة 10 صباحا
- مراجعة تقرير الساعة 2 مساءا


ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a54c-7103-ad95-2190b3ef3fde; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a54e-7d32-a7b3-9d1c68b8f690; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a54f-7e40-9b05-223334d3089e; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a551-7650-a8ec-eae080da0a7c; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a552-72f0-8c58-39331c4c5c99; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a41b-7440-a4da-072c9de2f321; trace=019f8bfc-a417-76f1-a7a6-010c9320390c,id=019f8bfc-a41b-7440-a4da-073256a22861


رد 2: نعم، يشمل هذا اليوم الأحد.


In [24]:
from langgraph.types import interrupt, Command

def email_agent_with_approval(state: MessagesState):
    response = llm_email.invoke(state["messages"])
    if response.tool_calls:
        approval = interrupt({"action": "إرسال إيميل", "details": response.tool_calls[0]["args"]})
        if approval != "approve":
            return {"messages": [("assistant", "تم إلغاء الإرسال بناءً على رفضك.")]}
    return {"messages": [response]}

builder4 = StateGraph(MessagesState)
builder4.add_node("email_agent", email_agent_with_approval)
builder4.add_node("email_tools", ToolNode(email_tools))
builder4.add_edge(START, "email_agent")
builder4.add_conditional_edges("email_agent", tools_condition, {"tools": "email_tools", END: END})
builder4.add_edge("email_tools", "email_agent")

hitl_graph = builder4.compile(checkpointer=MemorySaver())

config2 = {"configurable": {"thread_id": "hitl-1"}}
result = hitl_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]}, config=config2)
print(result)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8bfd-daa8-7fd3-8c33-f43a5c3c622c,id=019f8bfd-daa8-7fd3-8c33-f43a5c3c622c; trace=019f8bfd-daa8-7fd3-8c33-f43a5c3c622c,id=019f8bfd-daab-71b2-a5aa-0a0235c67688; trace=019f8bfd-daa8-7fd3-8c33-f43a5c3c622c,id=019f8bfd-daac-74a2-bb04-bc0f5fa7da72


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ky5y0y8zf8hskxhxernv52dt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99954, Requested 252. Please try again in 2m57.984s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [25]:
import time
time.sleep(180)  # ننتظر 3 دقايق

result = hitl_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]}, config=config2)
print(result)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c01-ad5c-79a2-98dd-514639542094,id=019f8c01-ad5c-79a2-98dd-514639542094; trace=019f8c01-ad5c-79a2-98dd-514639542094,id=019f8c01-ad5f-7ee0-8df5-481bce09d26f; trace=019f8c01-ad5c-79a2-98dd-514639542094,id=019f8c01-ad60-7c33-8b06-5d5bdfe4d69b


{'messages': [HumanMessage(content='ارسل ايميل لسارة بعنوان اجتماع الغد', additional_kwargs={}, response_metadata={}, id='a1cc49b3-28b1-436a-aeb0-5b42a5680fa0'), HumanMessage(content='ارسل ايميل لسارة بعنوان اجتماع الغد', additional_kwargs={}, response_metadata={}, id='a8c4b1d7-4064-4098-afde-8e11b1e9d36f')], '__interrupt__': [Interrupt(value={'action': 'إرسال إيميل', 'details': {'subject': 'اجتماع الغد', 'to': 'سارة'}}, id='9fc664876636f75bb50b6f103d60cdbd')]}


In [28]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print(llm.invoke("قول مرحبا").content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c06-4a4c-7eb0-9fc6-e7f6f7d4d86c,id=019f8c06-4a4c-7eb0-9fc6-e7f6f7d4d86c


مرحبا! كيف حالك؟


In [29]:
llm_calendar = llm.bind_tools(calendar_tools)
llm_email = llm.bind_tools(email_tools)
llm_rag = llm.bind_tools(rag_tools)

In [30]:
import time
time.sleep(20)

final_result = hitl_graph.invoke(Command(resume="approve"), config=config2)
print(final_result["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c07-2436-76c0-9f7e-f0d30bb2e644,id=019f8c07-2436-76c0-9f7e-f0d30bb2e644; trace=019f8c07-2436-76c0-9f7e-f0d30bb2e644,id=019f8c07-2438-7a92-9fd2-f9c80352e5ef; trace=019f8c07-2436-76c0-9f7e-f0d30bb2e644,id=019f8c07-2439-7c73-884b-8816815cf6b8


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=send_email>{"to": "sarah@example.com", "subject": " اجتماع الغد "</function>'}}

In [31]:
hitl_graph = builder4.compile(checkpointer=MemorySaver())
config2 = {"configurable": {"thread_id": "hitl-2"}}

result = hitl_graph.invoke({"messages": [("user", "ارسل ايميل لسارة بعنوان اجتماع الغد")]}, config=config2)
print(result)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c07-f941-7de1-8d47-4685ecdc4ad5,id=019f8c07-f941-7de1-8d47-4685ecdc4ad5; trace=019f8c07-f941-7de1-8d47-4685ecdc4ad5,id=019f8c07-f945-75b1-8273-af273e4121fe; trace=019f8c07-f941-7de1-8d47-4685ecdc4ad5,id=019f8c07-f948-7943-ab07-f2721e807a0e


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=send_email>{"subject": " اجتماع الغد","to": "سارة@example.com"}'}}

In [32]:
llm_calendar = llm.bind_tools(calendar_tools)
llm_email = llm.bind_tools(email_tools)
llm_rag = llm.bind_tools(rag_tools)

def email_agent_with_approval(state: MessagesState):
    for _ in range(3):
        try:
            response = llm_email.invoke(state["messages"])
            break
        except Exception:
            continue
    if response.tool_calls:
        approval = interrupt({"action": "إرسال إيميل", "details": response.tool_calls[0]["args"]})
        if approval != "approve":
            return {"messages": [("assistant", "تم إلغاء الإرسال.")]}
    return {"messages": [response]}

builder4 = StateGraph(MessagesState)
builder4.add_node("email_agent", email_agent_with_approval)
builder4.add_node("email_tools", ToolNode(email_tools))
builder4.add_edge(START, "email_agent")
builder4.add_conditional_edges("email_agent", tools_condition, {"tools": "email_tools", END: END})
builder4.add_edge("email_tools", "email_agent")
hitl_graph = builder4.compile(checkpointer=MemorySaver())

config3 = {"configurable": {"thread_id": "hitl-3"}}
result = hitl_graph.invoke({"messages": [("user", "send email to sara subject meeting tomorrow")]}, config=config3)
print(result)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c08-9bed-7840-a920-259aa280723d,id=019f8c08-9bed-7840-a920-259aa280723d; trace=019f8c08-9bed-7840-a920-259aa280723d,id=019f8c08-9bf5-7d33-b9fa-c4a6d4aa6ff3; trace=019f8c08-9bed-7840-a920-259aa280723d,id=019f8c08-9bf8-7462-b9ea-d701ffe39179


{'messages': [HumanMessage(content='send email to sara subject meeting tomorrow', additional_kwargs={}, response_metadata={}, id='29ee6b16-8401-4161-8925-3507465004d6')], '__interrupt__': [Interrupt(value={'action': 'إرسال إيميل', 'details': {'subject': 'meeting tomorrow', 'to': 'sara@example.com'}}, id='aa74d7e643ade78987e995adaba71cf0')]}


In [33]:
final_result = hitl_graph.invoke(Command(resume="approve"), config=config3)
print(final_result["messages"][-1].content)

ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0b78-7783-8560-3c3e8e756556; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0b7b-7713-b661-00fbae39e26a; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0b7c-7541-951c-99db8c3179ec
ضع_مفتاح_LangSmith_هنا
0
2
ordinal not in range(256)trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0cb1-7033-a5e3-d966af8eb325; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0cb2-75a3-8c55-ab90ade2ce55; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0cb4-7411-8ef0-de044bb54dcc; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0cb7-7be3-b941-439ca2d3672f; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0cb8-7013-b259-807033fc5ca1; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0b7b-7713-b661-00fbae39e26a; trace=019f8c09-0b78-7783-8560-3c3e8e756556,id=019f8c09-0b7c-7541-951c-99db8c3179ec


تم إرسال إيميل إلى sara@example.com بعنوان: meeting tomorrow


In [34]:
from langgraph.func import task, entrypoint

@task
def get_calendar_task(date: str) -> str:
    return get_calendar_events.invoke({"date": date})

@entrypoint(checkpointer=MemorySaver())
def calendar_workflow(date: str) -> str:
    result = get_calendar_task(date).result()
    return result

print(calendar_workflow.invoke("2026-07-24", config={"configurable": {"thread_id": "func-1"}}))

مواعيد 2026-07-24: ['اجتماع فريق 10ص', 'مراجعة تقرير 2م']


# Capstone Project — Building Agentic AI Systems

**Team Members:**
- Lama Hassan Alomrani
- Leena Mohammed Almalki

**Track:** A - Personal Assistant with Subagents

## Write-up

**Agent fundamentals:** بنينا agents حقيقية تستدعي أدوات فعلية (calendar, email, policy search) عبر bind_tools، بدون أي مخرجات ثابتة (hardcoded).

**Multi-agent/Routing:** استخدمنا نمط Supervisor يوجّه الطلب بين Calendar Agent و Email Agent حسب محتوى الرسالة.

**RAG pipeline:** طبّقنا 2-Step RAG: تحميل مستند سياسات → تقسيم بـ RecursiveCharacterTextSplitter → embedding بـ HuggingFace → تخزين بـ Chroma → استرجاع عبر tool. اخترنا 2-Step لبساطة المصدر (مستند واحد ثابت).

**Context & state:** استخدمنا MemorySaver كـ checkpointer مع thread_id، فحافظ الـ agent على سياق المحادثة عبر رسائل متعددة.

**Human-in-the-loop:** أضفنا interrupt() قبل تنفيذ إرسال الإيميل، يوقف التنفيذ وينتظر موافقة صريحة قبل الاستمرار عبر Command(resume=...).

**Error handling:** طبّقنا (1) transient retry عند مواجهة RateLimitError من Groq API، و(2) LLM-recoverable loopback حين فشل النموذج الصغير بصياغة tool call صحيح.

**Workflow pattern:** طبّقنا نمط Routing عبر الـ supervisor الذي يوجّه الطلبات لأحد الـ sub-agents.

**LangSmith:** فعّلنا tracing، وواجهنا مشكلة ترميز مع النصوص العربية أثناء رفع الـ traces.